# ZMART target acquisition — React edition

The same run as `zmart_microscopy_v4.ipynb`, with the review steps as live React apps inside the cells: every tile, focus point, and image pair appears the moment the microscope produces it, and the buttons in the browser drive the kernel (which drives the hardware through the same gated paths as always).

Needs the `anywidget` package in the kernel — nothing else. React ships inside the package (the official MIT-licensed build, evaluated privately in the browser), so this notebook works fully offline and fetches nothing from any CDN. Run the cells top to bottom.

## 1 · Setup and connect
Edit the analysis repo, then run.

In [ ]:
import sys
from pathlib import Path as _Path
_target_acq = _Path("workflows/target_acquisition")
if not _target_acq.is_dir():
    _target_acq = _Path.cwd()
sys.path.insert(0, str(_target_acq.resolve()))
from _bootstrap import Path, workflow
from workflow import react as wreact

ANALYSIS_REPO = Path("C:/code/smart-analysis")
SIMULATE_IMAGES = False
AUTOFOCUS_JOB = None  # Set only when LAS X has more than one autofocus job.

if "engine" in globals():
    try:
        engine.shutdown()
    except Exception as exc:
        print(f"warning: the previous analysis engine did not shut down cleanly: {exc}")
    del engine
if "zmart_controller" in globals():
    try:
        zmart_controller.disconnect()
    except Exception as exc:
        print(f"warning: the previous session did not disconnect cleanly: {exc}")
    del zmart_controller

engine = workflow.load_analysis_engine(ANALYSIS_REPO)
try:
    workflow.preflight_analysis_engine(engine)
    zmart_controller = workflow.connect("leica")
    ROOT = Path(zmart_controller.run_procedure({"name": "get_root"})["root"])
except Exception:
    try:
        if "zmart_controller" in globals():
            zmart_controller.disconnect()
            del zmart_controller
    finally:
        engine.shutdown()
        del engine
    raise
ROOT

### Where the run stands

A one-glance checklist built from the notebook's own variables — re-run this cell (or call `status.refresh(globals())`) after any step to see what is done, what is still to do, and what deserves a second look. It never talks to the microscope.

In [ ]:
status = wreact.run_status(globals())
status

## 2 · Set origin
Move to the run origin first. This makes the current position `(0, 0, 0)`.

In [ ]:
zmart_controller.set_origin()

## 3a · Overview job

Select the low-magnification overview job in LAS X, then run.

In [ ]:
overview_state = zmart_controller.get_state()
limits = overview_state["observed"].get("limits")
if not limits or limits.get("is_fallback") or limits.get("source") != "machine":
    raise RuntimeError(
        f"machine-specific stage limits are not active (got {limits}); publish this "
        "machine's measured envelope with limits/notebooks/set_stage_limits.ipynb first"
    )
overview_state["observed"]

## 3b · Target job

Select the high-magnification target job in LAS X, then run.

In [ ]:
target_state = zmart_controller.get_state()
if target_state["changeable"].get("job") == overview_state["changeable"].get("job"):
    raise RuntimeError("the overview and target jobs are the same; select the high-magnification target job")
target_state["observed"]

## 4 · Initial positions
Read overview tile centres from the controller.

In [ ]:
zmart_controller.set_state(overview_state)
positions = zmart_controller.run_procedure({"name": "get_positions"})["positions"]
print(len(positions), "overview positions")

## 5 · Focus

Click the map to add focus points (points already placed in LAS X are pre-filled; click a point to remove it), then press **Measure focus**. The stage autofocuses at each point and the fitted focus map streams in live, refining after every point. The map is drawn with +x to the right and +y downwards, the same orientation as the overview map below.

Re-measuring after editing points only visits the new or moved points — everything already measured this session is reused. If the focus may have drifted (a long pause, a bumped stage), press **Measure fresh** instead: it forgets this session's measurements and re-drives every point. Once measured, the overview tile markers take the colour of the fitted focus map.

In [ ]:
picker = wreact.pick_focus_points(
    zmart_controller, positions, af_job=AUTOFOCUS_JOB
)
picker

In [ ]:
focus = picker.require_focus()
workflow.plot_focus_surface(focus, save_path=ROOT / "focus_surface.png", show=True)
print("focus model:", focus.model)

## 6 · Overview

Capture one overview at each position and watch the map grow: every tile appears at its real stage position the moment it is saved, while the microscope scans the rest. Drag the map to pan and scroll to zoom (the view stops auto-fitting once you do — press **Fit** to frame everything again); the readout under the map shows the cursor's frame position in micrometres. The controls on the right work per channel: the colour swatch cycles the palette, **on/off** toggles visibility, and the min/max boxes set the display range for brightness and contrast (the same idea as Fiji's B&C — type a value and press Enter or click away to apply).

The map carries a scale bar (bottom right) that stays a nice round length as you zoom, and while the scan runs the status line counts tiles and estimates the time left.

In [ ]:
viewer = wreact.view_overview()
display(viewer)
viewer.expect_tiles(len(positions))  # lets the map show progress + time left
overview_records = workflow.run_overview(
    zmart_controller, positions, state=overview_state, focus=focus,
    on_record=viewer.add_acquisition,
)
print(len(overview_records), "overviews captured")
if SIMULATE_IMAGES:
    n = workflow.hijack_if_simulating(overview_records, simulate=True)
    viewer.reload()  # the hijack rewrote the saved pixels
    print(n, "overview images replaced")

In [ ]:
overviews = workflow.overview_inputs_from_records(positions, overview_records, focus=focus)

## 7 · Discover targets

Segment the overview images, then decide which cells become targets in the explorer:

- the two dropdowns put any measured feature on the plot's x and y axes (position, area, brightness, ...);
- gate with the min/max boxes under the plot (thresholds on the current axes — press Enter or click away to apply) and/or by drawing a lasso around points with the mouse — gated points stay blue, the rest fade out;
- hover a point to see that cell's image crop in the side panel.

`explorer.gated` is the list the next step samples from.

The same cells also appear on the overview map above — blue when they pass the gate, grey when they do not — and recolour live as you edit the gate, so you can judge the gating against the sample itself. Hover a ring on the map to see that cell's crop.

**Picking specific cells**: click a dot here (or its ring on the map) to pick that exact cell — picked cells wear a white outline, and the acquisition step below grows an **Acquire selected** button. Click again to un-pick. Cells already acquired this session fill in green on both figures, so nothing gets imaged twice by accident. Hovering a cell highlights it on both figures at once.

In [ ]:
targets = workflow.discover_targets(engine, overviews)
print(len(targets), "targets discovered")

In [ ]:
explorer = wreact.explore_targets(targets, overviews)
# The same cells on the overview map, recoloured live as the gate changes.
viewer.show_targets(targets, explorer)
explorer

## 8 · Acquire targets

Type how many targets to acquire and press **Acquire**. That many cells are drawn at random from the gate, re-imaged at the target job, and shown below as pairs — the overview crop (left) and the new high-magnification image (right), both over the same physical window so the cell appears at the same scale in both.

While a run is under way, a **Cancel** button appears: it stops cleanly before the next target (nothing is committed; in Jupyter the click may only reach Python when the kernel comes up for air). After the run, mark each pair **✓ good** or **✗ bad** — that judgement is your QC record of the run; write it into the run folder with `gallery.save_curation(ROOT)`.

**Acquire selected** images exactly the cells you picked (each pick is re-checked against the gate first); the plain **Acquire** still draws a random sample — press Enter in the count box as a shortcut. While the run works, the status shows how many are done and about how long is left. Click any finished pair to enlarge it (Esc closes), and note the scale bar in the corner of every image.

In [ ]:
gallery = wreact.acquire_gallery(
    zmart_controller,
    explorer,
    overviews,
    state=target_state,
    focus=focus,
    after_acquire=lambda records: workflow.hijack_if_simulating(records, simulate=SIMULATE_IMAGES),
)
gallery

## 9 · Summary
Write the summary and run map.

In [ ]:
if not gallery.records:
    raise RuntimeError("no targets have been acquired; use the Acquire button first")

summary = workflow.write_run_report(
    ROOT,
    positions=positions,
    focus=focus,
    overview_records=overview_records,
    targets=gallery.picked,
)
summary

In [ ]:
try:
    if "engine" in globals():
        engine.shutdown()
        del engine
finally:
    zmart_controller.disconnect()
    del zmart_controller